# Embeddings From First Principles

Neural networks only work with numbers. Words are symbols. This notebook builds the bridge — from raw text to tokenization to learned embeddings — from scratch.

## The Obvious Approach — Just Use a Dictionary

The simplest idea: assign each word a unique number. But what happens when we encounter a word that's not in our dictionary?

In [1]:
# The whole-word approach: just assign each word a number
dictionary = ["cat", "dog", "car", "fish", "the", "sat", "on", "mat"]
word_to_id = {word: i for i, word in enumerate(dictionary)}

print(f"Our dictionary ({len(dictionary)} words):")
for word, idx in word_to_id.items():
    print(f"  {word:>4s} → {idx}")

# Now try to look up words not in the dictionary
test_words = ["cat", "unfairly", "ChatGPT", "running"]
print(f"\nLooking up words:")
for word in test_words:
    if word in word_to_id:
        print(f"  {word:>10s} → {word_to_id[word]}")
    else:
        print(f"  {word:>10s} → ??? (not in dictionary)")

print("\nAny word not in the dictionary is invisible to the model.")
print("And language keeps creating new words. The vocabulary will never be complete.")

Our dictionary (8 words):
   cat → 0
   dog → 1
   car → 2
  fish → 3
   the → 4
   sat → 5
    on → 6
   mat → 7

Looking up words:
         cat → 0
    unfairly → ??? (not in dictionary)
     ChatGPT → ??? (not in dictionary)
     running → ??? (not in dictionary)

Any word not in the dictionary is invisible to the model.
And language keeps creating new words. The vocabulary will never be complete.


## What's a Token?

Characters give us universal coverage but no meaning. Whole words give us meaning but break on anything unseen. Let's look at both extremes.

In [2]:
# Character-level tokenization
text = "the cat sat on the mat"
char_tokens = list(text.replace(" ", ""))
char_vocab = sorted(set(char_tokens))

print(f"Text: \"{text}\"")
print(f"Character tokens: {char_tokens}")
print(f"Vocabulary ({len(char_vocab)} tokens): {char_vocab}")
print(f"\nEvery possible word can be built from these. But does 'a' mean something?")
print("Does 't'? Individual characters carry no meaning.")

Text: "the cat sat on the mat"
Character tokens: ['t', 'h', 'e', 'c', 'a', 't', 's', 'a', 't', 'o', 'n', 't', 'h', 'e', 'm', 'a', 't']
Vocabulary (9 tokens): ['a', 'c', 'e', 'h', 'm', 'n', 'o', 's', 't']

Every possible word can be built from these. But does 'a' mean something?
Does 't'? Individual characters carry no meaning.


## BPE — Letting the Data Decide

BPE starts from characters and greedily merges the most frequent adjacent pair, over and over, until the vocabulary reaches a target size. Let's watch it happen step by step.

In [3]:
from collections import Counter

corpus = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat chased the fish",
    "the dog chased the cat",
    "a car drove on the road",
    "a truck drove on the highway",
    "the fish swam in the pond",
    "the cat and dog played outside",
    "a car and truck were parked",
    "the dog ran across the yard",
    "the cat ran across the room",
]

def train_bpe(corpus, num_merges=15):
    """Train BPE tokenizer and show each merge step."""
    # Step 0: split every word into characters
    word_freqs = Counter()
    for sentence in corpus:
        for word in sentence.lower().split():
            word_freqs[tuple(word)] += 1
    
    merge_rules = []
    
    print(f"Initial vocabulary: {sorted(set(c for word in word_freqs for c in word))}\n")
    
    for step in range(num_merges):
        # Count all adjacent pairs
        pair_counts = Counter()
        for word, freq in word_freqs.items():
            for i in range(len(word) - 1):
                pair_counts[(word[i], word[i+1])] += freq
        
        if not pair_counts:
            break
        
        # Find the most frequent pair
        best_pair = pair_counts.most_common(1)[0]
        pair, count = best_pair
        merged = pair[0] + pair[1]
        merge_rules.append(pair)
        
        print(f"Merge {step+1:>2d}: ('{pair[0]}', '{pair[1]}') → '{merged}'  (count: {count})")
        
        # Apply the merge to all words
        new_word_freqs = Counter()
        for word, freq in word_freqs.items():
            new_word = []
            i = 0
            while i < len(word):
                if i < len(word) - 1 and word[i] == pair[0] and word[i+1] == pair[1]:
                    new_word.append(merged)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word_freqs[tuple(new_word)] += freq
        word_freqs = new_word_freqs
    
    # Show final vocabulary
    vocab = set()
    for word in word_freqs:
        for token in word:
            vocab.add(token)
    
    print(f"\nFinal vocabulary ({len(vocab)} tokens): {sorted(vocab, key=lambda x: (len(x), x))}")
    print(f"Merge rules: {merge_rules}")
    
    return word_freqs, merge_rules, vocab

word_freqs, merge_rules, vocab = train_bpe(corpus, num_merges=15)

Initial vocabulary: ['a', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y']

Merge  1: ('t', 'h') → 'th'  (count: 17)
Merge  2: ('th', 'e') → 'the'  (count: 17)
Merge  3: ('a', 't') → 'at'  (count: 8)
Merge  4: ('r', 'o') → 'ro'  (count: 6)
Merge  5: ('c', 'at') → 'cat'  (count: 5)
Merge  6: ('o', 'n') → 'on'  (count: 5)
Merge  7: ('d', 'o') → 'do'  (count: 4)
Merge  8: ('do', 'g') → 'dog'  (count: 4)
Merge  9: ('e', 'd') → 'ed'  (count: 4)
Merge 10: ('a', 'r') → 'ar'  (count: 4)
Merge 11: ('a', 'n') → 'an'  (count: 4)
Merge 12: ('r', 'u') → 'ru'  (count: 3)
Merge 13: ('s', 'at') → 'sat'  (count: 2)
Merge 14: ('c', 'h') → 'ch'  (count: 2)
Merge 15: ('ch', 'a') → 'cha'  (count: 2)

Final vocabulary (33 tokens): ['a', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y', 'an', 'ar', 'at', 'ed', 'on', 'ro', 'ru', 'cat', 'cha', 'dog', 'sat', 'the']
Merge rules: [('t', 'h'), ('th', 'e'), ('a', 't

## The Ambiguity Problem

After training, our vocabulary contains overlapping tokens — "t", "h", "th", "the" might all exist. A word could theoretically be segmented multiple ways. The merge rules resolve this: applied in order, they produce exactly one tokenization per word.

In [4]:
def tokenize_bpe(word, merge_rules):
    """Tokenize a word using trained BPE merge rules (applied in order)."""
    tokens = list(word.lower())
    
    for pair in merge_rules:
        merged = pair[0] + pair[1]
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == pair[0] and tokens[i+1] == pair[1]:
                new_tokens.append(merged)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    
    return tokens

# Tokenize some words — same merge rules, deterministic output
test_words = ["the", "cat", "chased", "across", "truck", "highway", "outside"]
print("Tokenization using trained BPE merge rules:\n")
for word in test_words:
    tokens = tokenize_bpe(word, merge_rules)
    print(f"  {word:>10s} → {tokens}")

print("\nSame word, same merge rules, same tokens. Every time.")
print("The merge rules are what make the tokenizer deterministic.")

Tokenization using trained BPE merge rules:

         the → ['the']
         cat → ['cat']
      chased → ['cha', 's', 'ed']
      across → ['a', 'c', 'ro', 's', 's']
       truck → ['t', 'ru', 'c', 'k']
     highway → ['h', 'i', 'g', 'h', 'w', 'a', 'y']
     outside → ['o', 'u', 't', 's', 'i', 'd', 'e']

Same word, same merge rules, same tokens. Every time.
The merge rules are what make the tokenizer deterministic.


## From Tokens to Vectors — The One-Hot Dead End

Now we have tokens. The obvious next step: assign each token a unique index and create a binary vector. Let's see why this fails.

In [5]:
import numpy as np
from itertools import combinations

# A vocabulary of 4 tokens
tokens = ['cat', 'dog', 'car', 'fish']
V = len(tokens)

# One-hot encode each token
one_hot = {token: np.eye(V)[i] for i, token in enumerate(tokens)}

print("One-hot encoding:\n")
for token, vec in one_hot.items():
    print(f"  {token:>4s} = {vec.astype(int).tolist()}")

# Compute the distance between every pair
print("\nEuclidean distances between all pairs:\n")
for t1, t2 in combinations(tokens, 2):
    dist = np.linalg.norm(one_hot[t1] - one_hot[t2])
    print(f"  d({t1:>4s}, {t2:<4s}) = {dist:.4f}")

print(f"\nEvery pair is equidistant at sqrt(2) = {np.sqrt(2):.4f}.")
print("The representation carries no information about token relationships.")

ModuleNotFoundError: No module named 'numpy'

## Measuring Context Similarity — From Jaccard to Cosine

First attempt: Jaccard similarity. What fraction of context words do two words share? But this ignores frequency — a context word appearing once counts as much as one appearing 100 times.

In [ ]:
def get_context_windows(corpus, window_size=2):
    """Extract (center_word, context_word) pairs from the corpus."""
    pairs = []
    for sentence in corpus:
        words = sentence.lower().split()
        for i, center in enumerate(words):
            for j in range(max(0, i - window_size), min(len(words), i + window_size + 1)):
                if i != j:
                    pairs.append((center, words[j]))
    return pairs

pairs = get_context_windows(corpus, window_size=2)

# Show context windows for a few words
for target in ['cat', 'dog', 'car']:
    contexts = [ctx for center, ctx in pairs if center == target]
    print(f"'{target}' appears near: {contexts}")
    print()

# --- Attempt 1: Jaccard similarity (ignores frequency) ---
def jaccard_overlap(word1, word2, pairs):
    ctx1 = set(ctx for center, ctx in pairs if center == word1)
    ctx2 = set(ctx for center, ctx in pairs if center == word2)
    shared = ctx1 & ctx2
    union = ctx1 | ctx2
    return len(shared) / len(union) if union else 0

# --- Attempt 2: Cosine similarity on frequency vectors ---
def context_cosine(word1, word2, pairs):
    ctx1 = Counter(ctx for center, ctx in pairs if center == word1)
    ctx2 = Counter(ctx for center, ctx in pairs if center == word2)
    all_ctx = sorted(set(ctx1.keys()) | set(ctx2.keys()))
    vec1 = np.array([ctx1.get(w, 0) for w in all_ctx], dtype=float)
    vec2 = np.array([ctx2.get(w, 0) for w in all_ctx], dtype=float)
    dot = np.dot(vec1, vec2)
    norm = np.linalg.norm(vec1) * np.linalg.norm(vec2)
    return dot / (norm + 1e-10)

test_pairs = [('cat', 'dog'), ('cat', 'car'), ('car', 'truck'), ('dog', 'truck')]

print("Jaccard (ignores frequency) vs Cosine (uses frequency):\n")
print(f"  {'Pair':>18s}  {'Jaccard':>8s}  {'Cosine':>8s}")
print(f"  {'─'*18}  {'─'*8}  {'─'*8}")
for w1, w2 in test_pairs:
    j = jaccard_overlap(w1, w2, pairs)
    c = context_cosine(w1, w2, pairs)
    print(f"  {w1:>7s} <-> {w2:<7s}  {j:>8.3f}  {c:>8.3f}")

print("\nJaccard treats all context words equally.")
print("Cosine weights frequent co-occurrences more heavily.")
print("But these frequency vectors are huge and sparse.")
print("What if we could learn small, dense vectors instead?")

## Word2Vec — Skip-Gram vs CBOW

Two ways to operationalize the distributional hypothesis:

- **Skip-gram:** given center word, predict each context word separately. Full gradient per word.
- **CBOW:** given all context words (averaged), predict center word. Gradient diluted by 1/N.

We implement both, plus all ablation variants.

In [ ]:
import random

# Build vocabulary from corpus
all_words = set()
for sentence in corpus:
    all_words.update(sentence.lower().split())
vocab = sorted(all_words)
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
V = len(vocab)

# Show the training pairs for both approaches
sentence = "the cat sat on"
words = sentence.lower().split()
print(f"Sentence: \"{sentence}\"  (window=1)\n")

print("SKIP-GRAM pairs (center → context, each separate):")
for i, center in enumerate(words):
    for j in range(max(0, i-1), min(len(words), i+2)):
        if i != j:
            print(f"  ({center}, {words[j]})  →  dot(u_{center}, v_{words[j]})  →  full gradient to u_{center}")

print(f"\nCBOW pairs (context → center, averaged):")
for i, center in enumerate(words):
    ctx = [words[j] for j in range(max(0, i-1), min(len(words), i+2)) if j != i]
    avg_str = " + ".join([f"u_{c}" for c in ctx]) + f") / {len(ctx)}"
    print(f"  {{{', '.join(ctx)}}} → {center}  →  avg = ({avg_str}  →  1/{len(ctx)} gradient each")

## The Ablations

Three questions, three experiments:

1. **Does the sigmoid matter?** Train with sigmoid loss vs linear loss. If both learn the same structure, the sigmoid doesn't contaminate the geometry.
2. **Input vs output embeddings:** Do both vectors learn the same thing? If a word has one meaning, they should agree.
3. **One vector vs two:** Train with a single embedding per word (used for both center and context roles). Does it learn the same structure as the two-vector approach?

In [ ]:
def sigmoid(x):
    """Numerically stable sigmoid."""
    return np.where(x >= 0,
                    1 / (1 + np.exp(-x)),
                    np.exp(x) / (1 + np.exp(x)))

class Word2Vec:
    def __init__(self, vocab_size, embed_dim, use_sigmoid=True, single_embedding=False):
        self.W_in = np.random.randn(vocab_size, embed_dim) * 0.1
        self.single_embedding = single_embedding
        if single_embedding:
            self.W_out = self.W_in
        else:
            self.W_out = np.random.randn(vocab_size, embed_dim) * 0.1
        self.use_sigmoid = use_sigmoid
    
    def skipgram_step(self, center_idx, context_idx, neg_indices, lr):
        """Skip-gram: center predicts context. One pair at a time."""
        v_center = self.W_in[center_idx].copy()
        u_context = self.W_out[context_idx].copy()
        
        # Positive: pull center toward context
        pos_score = np.dot(v_center, u_context)
        if self.use_sigmoid:
            pos_sig = sigmoid(pos_score)
            pos_loss = -np.log(pos_sig + 1e-10)
            grad_center_pos = (pos_sig - 1) * u_context
            grad_context = (pos_sig - 1) * v_center
        else:
            pos_loss = (1 - pos_score) ** 2
            grad_center_pos = -2 * (1 - pos_score) * u_context
            grad_context = -2 * (1 - pos_score) * v_center
        
        # Negatives: push center away from random words
        neg_loss = 0
        grad_center_neg = np.zeros_like(v_center)
        for neg_idx in neg_indices:
            u_neg = self.W_out[neg_idx].copy()
            neg_score = np.dot(v_center, u_neg)
            if self.use_sigmoid:
                neg_sig = sigmoid(neg_score)
                neg_loss += -np.log(1 - neg_sig + 1e-10)
                grad_center_neg += neg_sig * u_neg
                self.W_out[neg_idx] -= lr * (neg_sig * v_center)
            else:
                neg_loss += neg_score ** 2
                grad_center_neg += 2 * neg_score * u_neg
                self.W_out[neg_idx] -= lr * (2 * neg_score * v_center)
        
        # Update center (input) and context (output)
        self.W_in[center_idx] -= lr * (grad_center_pos + grad_center_neg)
        if not self.single_embedding:
            self.W_out[context_idx] -= lr * grad_context
        
        return pos_loss + neg_loss
    
    def cbow_step(self, center_idx, context_indices, neg_indices, lr):
        """CBOW: averaged context predicts center. Gradient diluted by 1/N."""
        N = len(context_indices)
        
        # Average context input embeddings
        ctx_vecs = np.array([self.W_in[ci].copy() for ci in context_indices])
        avg_ctx = ctx_vecs.mean(axis=0)
        
        u_center = self.W_out[center_idx].copy()
        
        # Positive: pull averaged context toward center
        pos_score = np.dot(avg_ctx, u_center)
        if self.use_sigmoid:
            pos_sig = sigmoid(pos_score)
            pos_loss = -np.log(pos_sig + 1e-10)
            grad_avg = (pos_sig - 1) * u_center
            grad_center = (pos_sig - 1) * avg_ctx
        else:
            pos_loss = (1 - pos_score) ** 2
            grad_avg = -2 * (1 - pos_score) * u_center
            grad_center = -2 * (1 - pos_score) * avg_ctx
        
        # Negatives
        neg_loss = 0
        grad_avg_neg = np.zeros_like(avg_ctx)
        for neg_idx in neg_indices:
            u_neg = self.W_out[neg_idx].copy()
            neg_score = np.dot(avg_ctx, u_neg)
            if self.use_sigmoid:
                neg_sig = sigmoid(neg_score)
                neg_loss += -np.log(1 - neg_sig + 1e-10)
                grad_avg_neg += neg_sig * u_neg
                self.W_out[neg_idx] -= lr * (neg_sig * avg_ctx)
            else:
                neg_loss += neg_score ** 2
                grad_avg_neg += 2 * neg_score * u_neg
                self.W_out[neg_idx] -= lr * (2 * neg_score * avg_ctx)
        
        # Distribute gradient back to each context word: each gets 1/N
        total_grad_avg = grad_avg + grad_avg_neg
        for ci in context_indices:
            self.W_in[ci] -= lr * (total_grad_avg / N)
        
        if not self.single_embedding:
            self.W_out[center_idx] -= lr * grad_center
        
        return pos_loss + neg_loss

print("Word2Vec model defined with:")
print("  - Skip-gram: center predicts each context word separately")
print("  - CBOW: averaged context predicts center word")
print("  - Options: sigmoid/linear loss, single/two embeddings")

In [ ]:
def train_word2vec(corpus, mode='skipgram', embed_dim=20, window_size=2, n_neg=5,
                   lr=0.025, epochs=100, use_sigmoid=True, single_embedding=False, seed=42):
    """Train Word2Vec with skip-gram or CBOW."""
    np.random.seed(seed)
    random.seed(seed)
    
    all_words = set()
    for sentence in corpus:
        all_words.update(sentence.lower().split())
    vocab = sorted(all_words)
    word2idx = {w: i for i, w in enumerate(vocab)}
    idx2word = {i: w for w, i in word2idx.items()}
    V = len(vocab)
    
    # Build training data: for skip-gram, pairs; for CBOW, (center, [contexts])
    sg_pairs = []
    cbow_pairs = []
    for sentence in corpus:
        words = sentence.lower().split()
        for i, center in enumerate(words):
            ctx_indices = []
            for j in range(max(0, i - window_size), min(len(words), i + window_size + 1)):
                if i != j:
                    sg_pairs.append((word2idx[center], word2idx[words[j]]))
                    ctx_indices.append(word2idx[words[j]])
            if ctx_indices:
                cbow_pairs.append((word2idx[center], ctx_indices))
    
    model = Word2Vec(V, embed_dim, use_sigmoid=use_sigmoid, single_embedding=single_embedding)
    
    losses = []
    for epoch in range(epochs):
        epoch_loss = 0
        
        if mode == 'skipgram':
            random.shuffle(sg_pairs)
            for center_idx, context_idx in sg_pairs:
                neg_indices = []
                while len(neg_indices) < n_neg:
                    neg = random.randint(0, V - 1)
                    if neg != center_idx and neg != context_idx:
                        neg_indices.append(neg)
                loss = model.skipgram_step(center_idx, context_idx, neg_indices, lr)
                epoch_loss += loss
            n_steps = len(sg_pairs)
        else:  # CBOW
            random.shuffle(cbow_pairs)
            for center_idx, ctx_indices in cbow_pairs:
                neg_indices = []
                while len(neg_indices) < n_neg:
                    neg = random.randint(0, V - 1)
                    if neg != center_idx and neg not in ctx_indices:
                        neg_indices.append(neg)
                loss = model.cbow_step(center_idx, ctx_indices, neg_indices, lr)
                epoch_loss += loss
            n_steps = len(cbow_pairs)
        
        avg_loss = epoch_loss / n_steps
        losses.append(avg_loss)
        if (epoch + 1) % 25 == 0:
            print(f"  Epoch {epoch+1:>3d}/{epochs}: avg loss = {avg_loss:.4f}")
    
    return model, vocab, word2idx, idx2word, losses

# Train all variants
configs = [
    ("Skip-gram + sigmoid (standard)",     'skipgram', True,  False),
    ("Skip-gram + linear (sigmoid ablation)", 'skipgram', False, False),
    ("Skip-gram + single emb (emb ablation)", 'skipgram', True,  True),
    ("CBOW + sigmoid",                     'cbow',     True,  False),
]

models = {}
all_losses = {}
for name, mode, use_sig, single in configs:
    print(f"\n{'=' * 55}")
    print(f"  {name}")
    print(f"{'=' * 55}")
    m, vocab, word2idx, idx2word, l = train_word2vec(
        corpus, mode=mode, use_sigmoid=use_sig, single_embedding=single
    )
    models[name] = m
    all_losses[name] = l

In [ ]:
import matplotlib.pyplot as plt

# Loss curves for all variants
fig, axes = plt.subplots(1, len(configs), figsize=(5*len(configs), 4))
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']

for ax, (name, _, _, _), color in zip(axes, configs, colors):
    ax.plot(all_losses[name], linewidth=2, color=color)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Avg Loss')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## The Ablations

Four questions:
1. **Sigmoid vs linear:** does the sigmoid contaminate the geometry?
2. **Input vs output embeddings:** a word has one meaning — do both vectors agree?
3. **Single vs two embeddings:** is the two-vector approach necessary?
4. **Skip-gram vs CBOW:** does gradient dilution change what's learned?

In [ ]:
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)

def get_nearest_neighbor(word, embeddings, word2idx, vocab):
    stop = ['the', 'a', 'on', 'in', 'and', 'were']
    best_word, best_sim = "", -2
    for w2 in vocab:
        if w2 == word or w2 in stop or w2 not in word2idx:
            continue
        sim = cosine_sim(embeddings[word2idx[word]], embeddings[word2idx[w2]])
        if sim > best_sim:
            best_word, best_sim = w2, sim
    return best_word, best_sim

def compare_embeddings(emb_a, emb_b, name_a, name_b, word2idx, vocab):
    similar = [('cat','dog'), ('car','truck'), ('cat','fish'), ('ran','chased')]
    dissimilar = [('cat','car'), ('dog','truck'), ('cat','road'), ('fish','drove')]
    content = [w for w in vocab if w not in ['the','a','on','in','and','were']]
    
    print(f"\n  {'Pair':>18s}  {name_a:>10s}  {name_b:>10s}")
    print(f"  {'─'*18}  {'─'*10}  {'─'*10}")
    print("  Similar:")
    for w1, w2 in similar:
        if w1 in word2idx and w2 in word2idx:
            sa = cosine_sim(emb_a[word2idx[w1]], emb_a[word2idx[w2]])
            sb = cosine_sim(emb_b[word2idx[w1]], emb_b[word2idx[w2]])
            print(f"  {w1:>7s} <-> {w2:<7s}  {sa:>+10.3f}  {sb:>+10.3f}")
    print("  Dissimilar:")
    for w1, w2 in dissimilar:
        if w1 in word2idx and w2 in word2idx:
            sa = cosine_sim(emb_a[word2idx[w1]], emb_a[word2idx[w2]])
            sb = cosine_sim(emb_b[word2idx[w1]], emb_b[word2idx[w2]])
            print(f"  {w1:>7s} <-> {w2:<7s}  {sa:>+10.3f}  {sb:>+10.3f}")
    
    print(f"\n  {'Word':>10s}  {name_a+' NN':>12s}  {name_b+' NN':>12s}  {'Same?':>6s}")
    print(f"  {'─'*10}  {'─'*12}  {'─'*12}  {'─'*6}")
    for w in content:
        nn_a = get_nearest_neighbor(w, emb_a, word2idx, vocab)
        nn_b = get_nearest_neighbor(w, emb_b, word2idx, vocab)
        same = "YES" if nn_a[0] == nn_b[0] else "no"
        print(f"  {w:>10s}  {nn_a[0]:>12s}  {nn_b[0]:>12s}  {same:>6s}")

# Ablation 1: Sigmoid vs Linear
m_std = models["Skip-gram + sigmoid (standard)"]
m_lin = models["Skip-gram + linear (sigmoid ablation)"]
print("=" * 55)
print("  ABLATION 1: Sigmoid vs Linear Loss")
print("=" * 55)
compare_embeddings(m_std.W_in, m_lin.W_in, "Sigmoid", "Linear", word2idx, vocab)

## Ablation 2: Input vs Output Embeddings
A word has one meaning. Do both vectors learn the same structure?

In [ ]:
print("=" * 55)
print("  ABLATION 2: Input vs Output Embeddings")
print("=" * 55)
compare_embeddings(m_std.W_in, m_std.W_out, "Input", "Output", word2idx, vocab)

## Ablation 3: Two Vectors vs Single Vector
Is the two-vector approach necessary, or can a single embedding per word learn the same structure?

In [ ]:
m_single = models["Skip-gram + single emb (emb ablation)"]
print("=" * 55)
print("  ABLATION 3: Two Vectors vs Single Vector")
print("=" * 55)
compare_embeddings(m_std.W_in, m_single.W_in, "Two-vec", "Single", word2idx, vocab)

## Ablation 4: Skip-gram vs CBOW
Does CBOW's gradient dilution (1/N from averaging) change the learned structure?

In [ ]:
m_cbow = models["CBOW + sigmoid"]
print("=" * 55)
print("  ABLATION 4: Skip-gram vs CBOW")
print("=" * 55)
compare_embeddings(m_std.W_in, m_cbow.W_in, "Skip-gram", "CBOW", word2idx, vocab)

In [ ]:
from sklearn.decomposition import PCA

def plot_embeddings(embeddings, vocab, word2idx, title, ax):
    content_words = [w for w in vocab if w not in ['the', 'a', 'on', 'in', 'and', 'were']]
    indices = [word2idx[w] for w in content_words]
    vecs = embeddings[indices]
    
    pca = PCA(n_components=2)
    coords = pca.fit_transform(vecs)
    
    ax.scatter(coords[:, 0], coords[:, 1], s=80, alpha=0.7, edgecolors='black', linewidth=0.5)
    for i, word in enumerate(content_words):
        ax.annotate(word, (coords[i, 0], coords[i, 1]),
                   fontsize=9, fontweight='bold',
                   xytext=(5, 5), textcoords='offset points')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.0%})')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.0%})')
    ax.grid(True, alpha=0.3)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

plot_embeddings(m_std.W_in, vocab, word2idx, 'Skip-gram + sigmoid (input)', axes[0, 0])
plot_embeddings(m_std.W_out, vocab, word2idx, 'Skip-gram + sigmoid (output)', axes[0, 1])
plot_embeddings(m_lin.W_in, vocab, word2idx, 'Skip-gram + linear (input)', axes[0, 2])
plot_embeddings(m_single.W_in, vocab, word2idx, 'Skip-gram + single emb', axes[1, 0])
plot_embeddings(m_cbow.W_in, vocab, word2idx, 'CBOW + sigmoid (input)', axes[1, 1])
plot_embeddings(m_cbow.W_out, vocab, word2idx, 'CBOW + sigmoid (output)', axes[1, 2])

plt.suptitle('Do all approaches learn the same structure?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()